In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
# @Date    : 2022-03-14 22:19:13
# @Author  : Tong Guo
# @Version : $V1$——2026

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import datetime
import seaborn as sns

In [ ]:
# read info
df_raw = pd.read_excel('周计划.xlsx')
# clean data 去除空值
# 计算指标 （区间段，计算非空均值，统计非空值）
# 输出可视化图片 （score趋势图，Task完成情况图）

In [ ]:
# clean data
df_raw.fillna(0,inplace=True)
# 计算周数和月数

df_raw['星期几'] = df_raw['Time'].dt.weekday
df_raw['week_name'] = df_raw['Time'].dt.strftime("%V")
df_raw

## 1. 计算指标
### 1）区间选取
### 2）计算非0数值均值，数据等


In [ ]:
# 1. 找出时间段内信息
# 找月份之间信息
df_period = df_raw[df_raw['Time'].dt.month.isin(np.arange(4,5))]   
# 获取day之间数据
open_day = '2026-01-01'
close_day = '2026-09-05'
con1 = df_raw['Time']>=open_day
con2 = df_raw['Time']<close_day
df_period = df_raw[con1&con2]
df_period 

In [ ]:
# 2. 统计相关信息
# 基本信息  天数，总均值，task完成状况量
n_day = df_period.shape[0]
average_period = df_period['Score'].mean()
n_Task1 = np.count_nonzero(df_period['Task1'])
n_Task2 = np.count_nonzero(df_period['Task2'])
n_Task3 = np.count_nonzero(df_period['Task3'])
n_task1 = np.count_nonzero(df_period['task_1'])
n_task2 = np.count_nonzero(df_period['task_2'])
n_task3 = np.count_nonzero(df_period['task_3'])
n_life1 = np.count_nonzero(df_period['Life_1'])
n_life2 = np.count_nonzero(df_period['Life_2'])
n_life3 = np.count_nonzero(df_period['Life_3'])

List_Index = ['天数','平均分数','主要任务完成次数','能力提升次数','生活任务次数']
List_value = [n_day,average_period,n_Task1+n_Task2+n_Task3,n_task1+n_task2+n_task3,n_life1+n_life2+n_life3]
df_sum = pd.DataFrame([])
df_sum['Name'] = np.array(List_Index)
df_sum['Value'] = np.array(List_value)
df_sum

In [ ]:
# 3. 根据每日 Score 计算成长率
conditions = [
    df_period["Score"] < 80,

    (df_period["Score"] >= 80)
    & (df_period["Score"] < 110),

    (df_period["Score"] >= 110)
    & (df_period["Score"] < 140),

    df_period["Score"] >= 140
]

growth_rates = [
    -0.001,
     0.001,
     0.005,
     0.01
]
df_period["Daily_Growth"] = np.select(
    conditions,
    growth_rates,
    default=np.nan
)
initial_value = 100

df_period["Growth_Value"] = (
    initial_value
    * (1 + df_period["Daily_Growth"].fillna(0)).cumprod()
)

df_period["Cumulative_Growth_Pct"] = (
    df_period["Growth_Value"] / initial_value - 1
) * 100

## 2. 可视化展示图
### 1）分数效率展示图（bar图）
### 2）任务时间分配图（饼图）
### 3）年任务线图 （线性图，词云图）

### 2.1 分数效率展示图（bar图）

In [ ]:
large = 22; med = 16; small = 12; dpi = 300
params = {'axes.titlesize': large,
          'legend.fontsize': med,
          'figure.figsize': (16, 10),
          'axes.labelsize': med,
          'axes.titlesize': med,
          'xtick.labelsize': med,
          'ytick.labelsize': med,
          'figure.titlesize': large,
          'figure.dpi': dpi}
plt.rcParams.update(params) # set the params for all plot
plt.rcParams["font.sans-serif"]=["SimHei"] #设置字体
plt.rcParams["axes.unicode_minus"]=False #该语句解决图像中的“-”负号的乱码问题

In [ ]:
color_plate1 = ['burlywood','firebrick','k']
colors = sns.color_palette('Paired')

In [ ]:
# 只在制图时复制
df_plot = df_period.copy()

mean_period = df_sum.loc[1, "Value"].round(2)

current_value = df_plot["Growth_Value"].iloc[-1]
current_growth_pct = df_plot["Cumulative_Growth_Pct"].iloc[-1]

fig, (ax1, ax2) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(16, 12),
    dpi=300,
    facecolor="w",
    gridspec_kw={
        "height_ratios": [2, 1],
        "hspace": 0.12
    },
    sharex=True
)

# --------------------------------------------------
# 上图：每日分数
# --------------------------------------------------

ax1.plot(
    "Time",
    "Score",
    data=df_plot,
    color=color_plate1[0],
    marker="o",
    markersize=4,
    label=f"平均分 {mean_period}"
)

ax1.axhline(
    y=80,
    color=color_plate1[1],
    alpha=0.7,
    linewidth=1,
    linestyle="--",
    label="标准线 80"
)

ax1.axhline(
    y=40,
    color=color_plate1[1],
    alpha=0.7,
    linewidth=1,
    linestyle=":",
    label="休息线 40"
)

ax1.axhline(
    y=140,
    color=color_plate1[1],
    alpha=0.7,
    linewidth=1,
    linestyle="-.",
    label="成长线 140"
)

ax1.set_ylim(
    0,
    max(160, df_plot["Score"].max() * 1.1)
)

ax1.set_ylabel("Score")
ax1.tick_params(
    axis="y",
    labelsize=12
)

ax1.grid(
    axis="y",
    alpha=0.7
)

ax1.legend()

# --------------------------------------------------
# 下图：复利成长
# --------------------------------------------------

growth_label = (
    f"当前成长值 {current_value:.2f} | "
    f"累计成长 {current_growth_pct:+.2f}%"
)

ax2.plot(
    "Time",
    "Growth_Value",
    data=df_plot,
    color=color_plate1[2],
    linewidth=2,
    label=growth_label
)

ax2.axhline(
    y=initial_value,
    color="gray",
    alpha=0.7,
    linewidth=1,
    linestyle="--",
    label=f"初始值 {initial_value}"
)

ax2.fill_between(
    df_plot["Time"],
    initial_value,
    df_plot["Growth_Value"],
    alpha=0.15
)

ax2.set_ylabel("Growth Value")
ax2.set_xlabel("Time")

ax2.grid(
    axis="y",
    alpha=0.7
)

ax2.legend()

# --------------------------------------------------
# 标题和保存
# --------------------------------------------------

Title = "Time period:" + open_day + " to " + close_day

fig.suptitle(
    Title,
    fontsize=22
)

path = (
    "results_bild/"
    + open_day
    + " to "
    + close_day
)

plt.savefig(
    path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# 周分数曲线图 （bar图）
df_bar = df_period.groupby('week_name').agg('mean')

# plot
fig,ax = plt.subplots(figsize=(16,10),dpi=300,facecolor='w',edgecolor='b')
ax.vlines(df_bar.index,ymin=0, ymax=df_bar['Score'],color=colors, alpha=0.7, linewidth=20)

# basic decoration
plt.gca().set(ylim=(0,140),ylabel='Score')
plt.title('week average score',fontsize =18)
plt.xticks(df_bar.index,df_bar.index,fontsize =12); plt.yticks(fontsize =12)
plt.xlabel('Week number',fontsize =16)

# advance decoration
# 1. text
for i,counts in enumerate(df_bar['Score']):
    ax.text(i,counts+0.5,round(counts,2), horizontalalignment='center')

path = 'results_bild/week_average_' + open_day + ' to ' + close_day    
plt.savefig(path,dpi=300)

### 2.2 任务时间分配图（饼图）

In [ ]:
# Calculating the non-zero counts for the specified columns
non_zero_counts = df_period[['Task1', 'Task2', 'Task3', 'task_1', 'task_2', 'task_3', 'Life_1', 'Life_2', 'Life_3']].apply(lambda x: (x!=0).sum())
non_zero_counts
# Calculating the sums for the required categories
main_task_sum =non_zero_counts[0:3].sum()
skill_improvement_sum = non_zero_counts[3:6].sum()
life_task_sum =  non_zero_counts[6:9].sum()
non_zero_counts.values

In [ ]:
# 定义颜色
colors_p = ["#2c3e50", "#e74c3c", "#ecf0f1", "#3498db", "#2980b9"]

# The original data (inner ring)
category_names = ['主要任务次数', '能力提升次数', '生活任务次数']
category_sizes = [main_task_sum, skill_improvement_sum, life_task_sum]

# The subdivisions (outer ring)
subcategory_names = ['Task1', 'Task2', 'Task3', 'task_1(量化)', 'task_2(自媒体)', 'task_3', 'Life_1(休息)', 'Life_2(健身)', 'Life_3']
subcategory_sizes = non_zero_counts.values.tolist()

# Define colors for each category and subcategory
category_colors = ["#e74c3c","#3498db","#2980b9"]
subcategory_colors = ['#99ccff', '#ffcccc', '#3399ff', '#ff6666', '#99ffff', "#2c3e50", "#ecf0f1", "#2c3e50", '#ffcccc']

# Plot
fig, ax = plt.subplots(figsize=(10, 10), dpi=300, facecolor='w', edgecolor='b')

# Outer Ring (made thinner and with a larger white circle to create space between the rings)
ax.pie(subcategory_sizes, radius=1.1, colors=subcategory_colors, labels=subcategory_names,
       autopct='%1.1f%%', pctdistance=0.92, startangle=90, wedgeprops=dict(width=0.2, edgecolor='w'))

# Inner Ring (made thicker)
inner_pie = ax.pie(category_sizes, radius=0.7, colors=category_colors, 
       autopct='%1.1f%%', pctdistance=0.75, startangle=90, wedgeprops=dict(width=0.3, edgecolor='w'))

# Draw a white circle at the center to make it a donut
centre_circle = plt.Circle((0,0),0.4,fc='white')
fig.gca().add_artist(centre_circle)
# Adding a legend for the inner ring categories
plt.legend(inner_pie[0], category_names, title="Categories", loc="center left", bbox_to_anchor=(1, 0.7))

plt.tight_layout()
plt.savefig('results_bild/ring1',dpi=300, bbox_inches='tight')
plt.show()


### 2.3 年任务线图 （线性图，词云图）

In [ ]:
# ==== 1) 数据准备 ====
df = df_period.copy()
# 转成字符串做去空格，再和 0/空串比较，同时也排除真正的数值 0
s = df['重大事件']
s_stripped = s.astype(str).str.strip()

mask = s.notna() & (s_stripped != "") & (s_stripped != "0") & (s != 0)
df_clean = df.loc[mask].sort_values('Time').copy()

# 如果要画图：
df_clean['Time'] = pd.to_datetime(df_clean['Time'])
df_clean

In [ ]:
# ==== 2) 画图 ====
if df_clean.empty:
    print("没有可显示的重大事件（清洗后为空）。")
else:
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'SimSun']
    plt.rcParams['axes.unicode_minus'] = False

    fig, ax = plt.subplots(figsize=(14, 3.5))

    # --- 基线（整条时间线） ---
    ax.hlines(0, df_clean['Time'].min(), df_clean['Time'].max(), linewidth=2, color="black")

    # --- 时间点 ---
    ax.scatter(df_clean['Time'], [0]*len(df_clean), s=80, color="steelblue", zorder=3)

    # --- 标注：上方写事件，下方写日期 ---
    for t, txt in zip(df_clean['Time'], df_clean['重大事件']):
        # 上方事件
        ax.annotate(str(txt), xy=(t, 0), xytext=(0, 15),
                    textcoords='offset points', ha='center', va='bottom', fontsize=10)
        # 下方日期
        ax.annotate(t.strftime('%Y-%m-%d'), xy=(t, 0), xytext=(0, -15),
                    textcoords='offset points', ha='center', va='top', fontsize=9, color="gray")

    # 美化
    ax.set_ylim(-2, 2)              # 留出上下空间
    ax.set_yticks([])               # 不要 y 轴
    for sp in ['left', 'right', 'top', 'bottom']:
        ax.spines[sp].set_visible(False)   # 去掉四周边框，只留我们画的基线

    ax.set_xticks([])               # 去掉默认 x 轴刻度
    ax.set_title("里程碑时间轴", fontsize=14, pad=12)

    plt.tight_layout()
    plt.savefig('results_bild/timeline',dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
import jieba
from wordcloud import WordCloud


In [ ]:
# 取出“每日感悟”列并合并成字符串
text = " ".join(df_period["每日感悟"].dropna().astype(str))

# 不使用 jieba，直接使用原始文本（按字统计）
text_cut = text

# 读取停用词
with open("background_data/中文停用词表.txt", encoding="utf8") as f:
    text_stop = f.read().split("\n")

# 生成词云
word_cloud = WordCloud(
    font_path="simsun.ttc",
    background_color="white",
    stopwords=set(text_stop),
    width=800,
    height=600
)

word_cloud.generate(text_cut)

# 展示
plt.figure(figsize=(12, 8))
plt.imshow(word_cloud, interpolation="bilinear")
plt.axis("off")
plt.savefig('results_bild/wordcloud',dpi=300, bbox_inches='tight')
plt.show()

## Backup

In [ ]:
# 1. 取出“每日感悟”这一列，并合并为一个长字符串
text = " ".join(df_period["每日感悟"].dropna().astype(str))

# 2. 使用 jieba 分词
text_cut = jieba.lcut(text)
text_cut = " ".join(text_cut)

# 3. 读取停用词表
with open("background_data/中文停用词表.txt", encoding="utf8") as f:
    text_stop = f.read().split("\n")


In [ ]:
# 4. 生成词云
word_cloud = WordCloud(
    font_path="simsun.ttc",     # 确保本地有宋体或替换为你本地的中文字体路径
    background_color="white",
    stopwords=set(text_stop),   # 用 set 更好
    width=800,
    height=600
)

word_cloud.generate(text_cut)

# 5. 展示
plt.figure(figsize=(12, 8))
plt.imshow(word_cloud, interpolation="bilinear")
plt.axis("off")
plt.show()

In [ ]:
# 任务对比 （饼图）
# plot
df_task = df_sum.loc[2:,:]
data = df_task['Value']
categories = df_task['Name']


fig,ax = plt.subplots(figsize = (12,7),facecolor='w',edgecolor='b')
patches,l_text,p_text = ax.pie(data, labels=categories, autopct='%1.1f%%',textprops=dict(color="w"),colors=color_plate1,startangle=140)
# autopct 是在上面加文字的固定为'%1.1f%%'，textprops=dict(color="w") 为字的颜色
# startangle=140 为开始突出的角度，explode为第几个需要explode
# labels=categories 决定是否有外部标签，可取掉
for t in p_text:  # 饼图内部文本
    t.set_size(12)
for t in l_text:  # 饼图外部文本
    t.set_size(12)

# basic decoration
Title = 'Time period:' + open_day + ' to ' + close_day
plt.title(Title,fontsize=16)
plt.legend(categories,title='Task', loc='center right',bbox_to_anchor=(1, 0, 0.5, 1))  #bbox_to_anchor 具体位置
path = 'results_bild/task_complete' + open_day + ' to ' + close_day    
plt.savefig(path,dpi=300)


In [ ]:
# 天分数曲线图 （线图）
# plot
mean_period = df_sum.loc[1,'Value'].round(2)
fig,ax = plt.subplots(figsize=(16,9),dpi=300,facecolor='w',edgecolor='b')
ax.plot('Time','Score',data=df_period,color=color_plate1[0], marker='o',label='平均分 {}'.format(mean_period))
ax.hlines(y=80, xmin=df_period.Time.tolist()[0], xmax=df_period.Time.tolist()[-1], color=color_plate1[1], alpha=0.7, linewidth=1,label='标准线 80')
ax.hlines(y=40, xmin=df_period.Time.tolist()[0], xmax=df_period.Time.tolist()[-1], color=color_plate1[1], alpha=0.7, linewidth=1,label='标准线 40')
# basic decoration
plt.gca().set(ylim=(0,150),ylabel='Score')
# 横坐标太多，用该方法减少一些

plt.yticks(fontsize=12,alpha=0.7)
Title = 'Time period:' + open_day + ' to ' + close_day
plt.title(Title,fontsize=22)
plt.grid(axis='y',alpha=0.7)  # grid 可以只加横或纵，也能设置透明度
plt.legend()
#plt.gca().spines["top"].set_alpha(0.0)  # 可以去外框  
path = 'results_bild/' + open_day + ' to ' + close_day
plt.savefig(path,dpi=300)